# Assignment 1 — Gradio dashboard

You're using a notebook for this stage. If you'd rather use a script,
write `app.py` instead and delete this file — `main.py` refuses to run if
it finds both `app.py` and `app.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py app`, this notebook must build and launch a Gradio
app (a `demo` that calls `.launch()`) using only what your training stage
already produced — it must never retrain anything itself. At minimum, the
dashboard must let you:

- Compare your three trained models via a prediction-vs-actual plot.
- See feature and/or target distributions.
- For classification: move a decision-threshold slider and watch the
  confusion matrix, and a business-cost number tied to your `REPORT.md`'s
  framing, change with it.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

In [25]:

import json, joblib    
import numpy as np    
import pandas as pd      
import torch            
import torch.nn as nn  
import gradio as gr     
import plotly.express as px 
import plotly.graph_objects as go   
from plotly.subplots import make_subplots    

with open("artifacts/meta.json") as f:
    meta = json.load(f)                                      
scaler = joblib.load("artifacts/scaler.joblib")               
sk_model = joblib.load("artifacts/sklearn_model.joblib")      
manual = torch.load("artifacts/manual_torch.pt")             
results = pd.read_csv("artifacts/results.csv", index_col=0)
preds_df = pd.read_csv("artifacts/predictions.csv")


class LinearReg(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.linear = nn.Linear(n_inputs, 1)
    def forward(self, x):
        return self.linear(x)

nn_model = LinearReg(meta["n_features"])   
nn_model.load_state_dict(torch.load("artifacts/nn_torch.pt"))
nn_model.eval()

LinearReg(
  (linear): Linear(in_features=32, out_features=1, bias=True)
)

In [26]:
### 2. District list and the chart functions

d = pd.read_csv("data/listings.csv", usecols=["neighbourhood_group_cleansed", "room_type", "bedrooms", "price"])
d["price"] = d["price"].str.replace("[$,]", "", regex=True).astype(float)  # price text -> number
d = d[(d["room_type"] == "Entire home/apt") & (d["bedrooms"] == 2) & (d["price"] <= 500)]# same scope as training
all_districts = sorted(d["neighbourhood_group_cleansed"].dropna().unique())  # every district in the scope


dropped_district = [x for x in all_districts if "district_" + x not in meta["district_cols"]]
print("districts:", len(all_districts), "| district with no column:", dropped_district)

MODEL_COLS = {"scikit-learn": "pred_sklearn",  
              "Manual PyTorch": "pred_manual_torch",
              "nn.Module PyTorch": "pred_nn_torch"}

# View 1: predicted vs actual price, one panel per model 
def compare_plot(split):
    d_ = preds_df[preds_df["split"] == split]
    fig = make_subplots(rows=1, cols=3, subplot_titles=list(MODEL_COLS), shared_yaxes=True)
    for i, col in enumerate(MODEL_COLS.values(), start=1):
        fig.add_trace(go.Scatter(x=d_["price"], y=d_[col], mode="markers",
                                 marker=dict(size=4, opacity=0.5), showlegend=False), row=1, col=i) 
        fig.add_trace(go.Scatter(x=[0, 500], y=[0, 500], mode="lines",
                                 line=dict(dash="dash"), showlegend=False), row=1, col=i) 
        fig.update_xaxes(title_text="actual price (EUR)", row=1, col=i)
    fig.update_yaxes(title_text="predicted price (EUR)", row=1, col=1)
    fig.update_layout(height=450, template="plotly_white", title=f"Predicted vs actual price ({split} set, {len(d_)} flats)")   # white background, less grey
    return fig

# View 2: distribution of the price or of one feature, per data split
def dist_plot(feature):
    fig = px.histogram(preds_df, x=feature, color="split", nbins=25, barmode="group", histnorm="percent")   # side-by-side bars instead of muddy overlap
    fig.update_layout(height=450, template="plotly_white", title=f"Distribution of {feature}")
    return fig


districts: 21 | district with no column: ['Arganzuela']


The district list is for the dropdown in the “check my flat” form. The app needs the district names to let the host pick one.
View 1 is the required prediction-vs-actual plot: one dot per flat, with the dashed diagonal line showing a perfect prediction. Dots close to the line are good predictions.
View 2 is the required distribution plot. It compares the price (or a feature) across train, validation and test.
Nothing is trained here, and the functions only read the saved predictions.

In [ ]:
## 3. The "check my flat" tool

def check_flat(district, accommodates, beds, bathrooms, amenities, dist_km, min_nights,
               host_listings, n_reviews, reviews_pm, rating, loc_score, planned_price, x_pct):
    #A new listing has no review data, so use the same fills as in training 
    has_rev = 1 if n_reviews > 0 else 0
    if not has_rev:
        reviews_pm = 0.0              
        rating = meta["medians"]["review_scores_rating"]
        loc_score = meta["medians"]["review_scores_location"]


    row = {c: 0.0 for c in meta["feature_order"]} 
    row.update({"accommodates": accommodates, "beds": beds, "bathrooms_num": bathrooms,
                "amenity_count": amenities, "dist_center_km": dist_km,
                "calculated_host_listings_count": host_listings, "minimum_nights_avg_ntm": min_nights,
                "number_of_reviews": n_reviews, "reviews_per_month": reviews_pm,
                "review_scores_rating": rating, "review_scores_location": loc_score,
                "has_reviews": has_rev})
    if "district_" + district in row:
        row["district_" + district] = 1.0      
    X = pd.DataFrame([row])[meta["feature_order"]] 
    X[meta["chosen_numeric"]] = scaler.transform(X[meta["chosen_numeric"]]) 

    xt = torch.tensor(X.values, dtype=torch.float32)
    with torch.no_grad():
        p_sk = float(sk_model.predict(X)[0])                    # method 1
        p_man = float((xt @ manual["w"] + manual["b"]).item())  # method 2
        p_nn = float(nn_model(xt).item())                       # method 3

    # --- The "too much" rule: warn if the planned price is more than X% above the typical price ---
    limit = p_sk * (1 + x_pct / 100)                            # the line you must not cross
    miss = results.loc["sklearn", "test_MAE"]                   # the model's typical error on flats it has not seen

    # --- Pick a message depending on where the planned price lands ---
    if planned_price > limit:
        verdict = (f" **CAREFUL!** {planned_price:.0f} EUR is more than {x_pct:.0f}% above what similar flats "
                   f"usually charge. Guests will notice. Are you sure about that price?")
    elif planned_price > p_sk:
        verdict = (f"**CAREFUL! This pricing seems ambitious** {planned_price:.0f} EUR is above the typical price, "
                   f"yet still inside your {x_pct:.0f}% comfort zone.")
    elif planned_price >= 0.8 * p_sk:    
        verdict = (f"**GOOD CHOICE of pricing** {planned_price:.0f} EUR is in the normal range for a flat like yours")
    else:
        verdict = (f"** CAREFUL ! This is cheap pricing.** {planned_price:.0f} EUR is well below similar flats"
                   f"Maybe you could ask for a bit more?")

    return (f"## Flats like yours go for about {p_sk:.0f} EUR a night\n\n"
            f"{verdict}\n\n"
            f"The warning kicks in above **{limit:.0f} EUR**.\n\n"
            f"**Do the three models agree?** scikit-learn says {p_sk:.0f}, my hand-written PyTorch loop says {p_man:.0f}, "
            f"and the `nn.Module` version says {p_nn:.0f}.\n\n"
            f"**Be careful:** on flats it had never seen, this model was off by about {miss:.0f} EUR on average, "
            f"so use it as a rough guide, not a verdict. It only knows entire 2-bedroom homes up to 500 EUR a night, "
            f"so a penthouse or a single room is outside what it knows")

## Flats like yours go for about 224 EUR a night

**GOOD CHOICE of pricing** 200 EUR is in the normal range for a flat like yours

The warning kicks in above **336 EUR**.

**Do the three models agree?** scikit-learn says 224, my hand-written PyTorch loop says 224, and the `nn.Module` version says 224.

**Be careful:** on flats it had never seen, this model was off by about 62 EUR on average, so use it as a rough guide, not a verdict. It only knows entire 2-bedroom homes up to 500 EUR a night, so a penthouse or a single room is outside what it knows


La idea: A new flat goes through the same steps as in training: fill the review fields the same way, build the same 32 columns in the same order, scale with the saved scaler, then predict. If any step were different, the predictions would be wrong.
It predicts with all three saved models, which is why the app can show they agree.
It trains nothing and fits nothing, which is what the README requires.

R² is only 0.29, which is weak. The strongest single
correlation was 0.34, and what information about the flats is not in the data?).
The model still beats the baseline on the test set (about 62 vs 71 EUR), so it
has learned something. I haven't compared it to other models, so I can't say how
it compares with others.

A new flat goes through the same steps as in training (same fills, same 32 columns, same scaling), then the three saved models predict.

In [28]:
# 4. build dashboard and launch it 

with gr.Blocks() as demo:
    gr.Markdown("# Madrid Airbnb pricing tool\n"
                "What does a normal 2-bedroom flat in Madrid charge per night? "
                "Models trained on Inside Airbnb data (September 2026). Nothing is retrained here.")

    #TAB 1: the three models compared
    with gr.Tab("Compare the three models"):
        gr.Dataframe(value=results.round(2).rename_axis("method").reset_index(), interactive=False)   # MAE in EUR, R2
        split_in = gr.Radio(["val", "test"], value="val", label="Which set to plot")
        plot_out = gr.Plot()
        split_in.change(compare_plot, split_in, plot_out) 
        demo.load(compare_plot, split_in, plot_out) 

    #TAB 2: distributions of the price and the features 
    with gr.Tab("Distributions"):
        feat_in = gr.Dropdown(["price"] + meta["chosen_numeric"], value="price", label="Price or feature")
        dist_out = gr.Plot()
        feat_in.change(dist_plot, feat_in, dist_out)
        demo.load(dist_plot, feat_in, dist_out)

    #TAB 3: check my flat
    with gr.Tab("Check my flat"):
        district = gr.Dropdown(all_districts, value="Centro" if "Centro" in all_districts else all_districts[0], label="District")
        with gr.Row():
            accommodates = gr.Number(value=4, label="Guests it sleeps")
            beds = gr.Number(value=3, label="Beds")
            bathrooms = gr.Number(value=1, label="Bathrooms")
            amenities = gr.Number(value=30, label="Number of amenities listed")
        with gr.Row():
            dist_km = gr.Number(value=2.0, label="Distance to Puerta del Sol (km)")
            min_nights = gr.Number(value=2, label="Minimum nights")
            host_listings = gr.Number(value=1, label="Listings you own")
        with gr.Row():
            n_reviews = gr.Number(value=0, label="Number of reviews (0 if new)")
            reviews_pm = gr.Number(value=0, label="Reviews per month")
            rating = gr.Number(value=meta["medians"]["review_scores_rating"], label="Overall rating")
            loc_score = gr.Number(value=meta["medians"]["review_scores_location"], label="Location rating")
        planned = gr.Number(value=200, label="Your planned price (EUR per night)")
        x_slider = gr.Slider(10, 100, value=50, step=5, label="Warn me if my price is more than X% above typical")
        btn = gr.Button("Check my price")
        out = gr.Markdown()
        btn.click(check_flat,
                  inputs=[district, accommodates, beds, bathrooms, amenities, dist_km, min_nights,
                          host_listings, n_reviews, reviews_pm, rating, loc_score, planned, x_slider],
                  outputs=out)

demo.launch(inbrowser=True) 


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
